# 04 · Data Relationships

**Project:** Enterprise HR AI  
**Parts:**
- **A** — Lightweight cleaning of three O\*NET reference files → `data/processed/`
- **B** — Formal join-coverage confirmation between processed attrition & engagement tables
- **C** — Full relationship map → `docs/data_relationships.md`

**Rule:** No merges performed. Analysis only. All decisions printed explicitly.

---

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

RAW  = os.path.join('..', 'data', 'raw')
PROC = os.path.join('..', 'data', 'processed')
DOCS = os.path.join('..', 'docs')
os.makedirs(DOCS, exist_ok=True)

print('RAW  :', os.path.abspath(RAW))
print('PROC :', os.path.abspath(PROC))
print('DOCS :', os.path.abspath(DOCS))

RAW  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\raw
PROC : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed
DOCS : C:\Users\ASUS\Desktop\enterprise_hr_ai\docs


---
## Part A · Reference File Cleaning

Lightweight cleaning for the three O\*NET master/reference files.  
**Rule:** Strip whitespace, drop exact duplicates, confirm key columns have no nulls.  
Do NOT correct values — these are master data, not employee records.

---

### A1 · occupation_data.csv

In [2]:
occ_raw = pd.read_csv(os.path.join(RAW, 'occupation_data.csv'))
occ = occ_raw.copy()
print(f'Loaded occupation_data.csv: {occ.shape[0]:,} rows x {occ.shape[1]} cols')
print(f'Columns: {occ.columns.tolist()}')

Loaded occupation_data.csv: 1,016 rows x 3 cols
Columns: ['O*NET-SOC Code', 'Title', 'Description']


In [3]:
# 1. Strip whitespace on all object columns
for col in occ.select_dtypes(include='object').columns:
    occ[col] = occ[col].str.strip()
print('Whitespace stripped on all object columns.')

# 2. Drop exact duplicate rows
n_before = len(occ)
occ.drop_duplicates(inplace=True)
occ.reset_index(drop=True, inplace=True)
n_after = len(occ)
print(f'Duplicates dropped: {n_before - n_after}  ({n_before} -> {n_after} rows)')

# 3. Null check on key identifying columns
KEY_COLS_OCC = ['O*NET-SOC Code', 'Title']
for col in KEY_COLS_OCC:
    nulls = occ[col].isnull().sum()
    status = 'OK (0 nulls)' if nulls == 0 else f'WARNING: {nulls} nulls'
    print(f'  [{col}]: {status}')

Whitespace stripped on all object columns.
Duplicates dropped: 0  (1016 -> 1016 rows)
  [O*NET-SOC Code]: OK (0 nulls)
  [Title]: OK (0 nulls)


In [4]:
occ_out = os.path.join(PROC, 'occupation_master.csv')
occ.to_csv(occ_out, index=False)
print(f'Saved: occupation_master.csv  ({occ.shape[0]:,} rows x {occ.shape[1]} cols)')

Saved: occupation_master.csv  (1,016 rows x 3 cols)


### A2 · essential_skills.csv

In [5]:
ess_raw = pd.read_csv(os.path.join(RAW, 'essential_skills.csv'))
ess = ess_raw.copy()
print(f'Loaded essential_skills.csv: {ess.shape[0]:,} rows x {ess.shape[1]} cols')
print(f'Columns: {ess.columns.tolist()}')

Loaded essential_skills.csv: 18,200 rows x 15 cols
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']


In [6]:
# 1. Strip whitespace on all object columns
for col in ess.select_dtypes(include='object').columns:
    ess[col] = ess[col].str.strip()
print('Whitespace stripped on all object columns.')

# 2. Drop exact duplicate rows
n_before = len(ess)
ess.drop_duplicates(inplace=True)
ess.reset_index(drop=True, inplace=True)
n_after = len(ess)
print(f'Duplicates dropped: {n_before - n_after}  ({n_before} -> {n_after} rows)')

# 3. Null check on key columns
KEY_COLS_ESS = ['O*NET-SOC Code', 'Title']
for col in KEY_COLS_ESS:
    nulls = ess[col].isnull().sum()
    status = 'OK (0 nulls)' if nulls == 0 else f'WARNING: {nulls} nulls'
    print(f'  [{col}]: {status}')

# 4. Also print null status for 'Not Relevant' (known 50% missing from Step 1)
nr_nulls = ess['Not Relevant'].isnull().sum()
print(f'  [Not Relevant]: {nr_nulls} nulls ({nr_nulls/len(ess)*100:.1f}%)'
      f' -- expected (IM rows have no Not-Relevant flag), NOT a data defect')

Whitespace stripped on all object columns.
Duplicates dropped: 0  (18200 -> 18200 rows)
  [O*NET-SOC Code]: OK (0 nulls)
  [Title]: OK (0 nulls)
  [Not Relevant]: 9100 nulls (50.0%) -- expected (IM rows have no Not-Relevant flag), NOT a data defect


In [7]:
ess_out = os.path.join(PROC, 'essential_skills_processed.csv')
ess.to_csv(ess_out, index=False)
print(f'Saved: essential_skills_processed.csv  ({ess.shape[0]:,} rows x {ess.shape[1]} cols)')

Saved: essential_skills_processed.csv  (18,200 rows x 15 cols)


### A3 · software_skills.csv

In [8]:
sw_raw = pd.read_csv(os.path.join(RAW, 'software_skills.csv'))
sw = sw_raw.copy()
print(f'Loaded software_skills.csv: {sw.shape[0]:,} rows x {sw.shape[1]} cols')
print(f'Columns: {sw.columns.tolist()}')

Loaded software_skills.csv: 31,821 rows x 7 cols
Columns: ['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']


In [9]:
# 1. Strip whitespace on all object columns
for col in sw.select_dtypes(include='object').columns:
    sw[col] = sw[col].str.strip()
print('Whitespace stripped on all object columns.')

# 2. Drop exact duplicate rows
n_before = len(sw)
sw.drop_duplicates(inplace=True)
sw.reset_index(drop=True, inplace=True)
n_after = len(sw)
print(f'Duplicates dropped: {n_before - n_after}  ({n_before} -> {n_after} rows)')

# 3. Null check on key columns
KEY_COLS_SW = ['O*NET-SOC Code', 'Title']
for col in KEY_COLS_SW:
    nulls = sw[col].isnull().sum()
    status = 'OK (0 nulls)' if nulls == 0 else f'WARNING: {nulls} nulls'
    print(f'  [{col}]: {status}')

Whitespace stripped on all object columns.
Duplicates dropped: 0  (31821 -> 31821 rows)
  [O*NET-SOC Code]: OK (0 nulls)
  [Title]: OK (0 nulls)


In [10]:
sw_out = os.path.join(PROC, 'software_skills_processed.csv')
sw.to_csv(sw_out, index=False)
print(f'Saved: software_skills_processed.csv  ({sw.shape[0]:,} rows x {sw.shape[1]} cols)')

Saved: software_skills_processed.csv  (31,821 rows x 7 cols)


In [11]:
print('=== PART A SUMMARY — Reference File Cleaning ===')
for label, raw_df, clean_df in [
    ('occupation_master',            occ_raw, occ),
    ('essential_skills_processed',   ess_raw, ess),
    ('software_skills_processed',    sw_raw,  sw),
]:
    dropped = raw_df.shape[0] - clean_df.shape[0]
    print(f'  {label:<35s}: {raw_df.shape[0]:>6,} -> {clean_df.shape[0]:>6,} rows  '
          f'(dropped {dropped} duplicates)')

=== PART A SUMMARY — Reference File Cleaning ===
  occupation_master                  :  1,016 ->  1,016 rows  (dropped 0 duplicates)
  essential_skills_processed         : 18,200 -> 18,200 rows  (dropped 0 duplicates)
  software_skills_processed          : 31,821 -> 31,821 rows  (dropped 0 duplicates)


---
## Part B · Join-Coverage Confirmation

Using the **processed** files. No merge performed — analysis only.

---

In [12]:
att = pd.read_csv(os.path.join(PROC, 'employee_attrition_processed.csv'))
eng = pd.read_csv(os.path.join(PROC, 'engagement_processed.csv'))

print(f'employee_attrition_processed : {att.shape[0]:,} rows x {att.shape[1]} cols')
print(f'engagement_processed         : {eng.shape[0]:,} rows x {eng.shape[1]} cols')

employee_attrition_processed : 1,470 rows x 35 cols
engagement_processed         : 2,845 rows x 28 cols


In [13]:
# Key columns
ATT_KEY = 'EmployeeNumber'
ENG_KEY = 'Employee ID'

att_ids = set(att[ATT_KEY].dropna().astype(int))
eng_ids = set(eng[ENG_KEY].dropna().astype(int))

both       = att_ids & eng_ids          # in both
att_only   = att_ids - eng_ids          # in attrition only
eng_only   = eng_ids - att_ids          # in engagement only

n_att      = len(att_ids)
n_eng      = len(eng_ids)
n_both     = len(both)
n_att_only = len(att_only)
n_eng_only = len(eng_only)

pct_of_att = n_both / n_att * 100
pct_of_eng = n_both / n_eng * 100

print('=== JOIN COVERAGE — PROCESSED FILES ===')
print()
print(f'Total unique EmployeeNumber in attrition : {n_att:,}')
print(f'Total unique Employee ID    in engagement: {n_eng:,}')
print()
print('--- 2x2 Breakdown ---')
print(f'  (a) In BOTH files        : {n_both:,}  ({pct_of_att:.1f}% of attrition, {pct_of_eng:.1f}% of engagement)')
print(f'  (b) Attrition only       : {n_att_only:,}  (no engagement record)')
print(f'  (c) Engagement only      : {n_eng_only:,}  (no attrition record)')
print(f'  (d) Total union          : {len(att_ids | eng_ids):,}')
print()
print(f'Match vs Step-1 finding (731/1470 = 49.7%): '
      f'{n_both}/{n_att} = {pct_of_att:.1f}%  -> '
      f'{"CONFIRMED" if n_both == 731 else "CHANGED -- investigate"}')

=== JOIN COVERAGE — PROCESSED FILES ===

Total unique EmployeeNumber in attrition : 1,470
Total unique Employee ID    in engagement: 2,845

--- 2x2 Breakdown ---
  (a) In BOTH files        : 731  (49.7% of attrition, 25.7% of engagement)
  (b) Attrition only       : 739  (no engagement record)
  (c) Engagement only      : 2,114  (no attrition record)
  (d) Total union          : 3,584

Match vs Step-1 finding (731/1470 = 49.7%): 731/1470 = 49.7%  -> CONFIRMED


### Join Decision

> **DECISION:** `employee_attrition` and `engagement` data have a **49.7% overlap (731 of 1,470 attrition employees have engagement records).**  
>  
> We treat `employee_attrition_processed.csv` as the **ANCHOR table** for all attrition modelling (Day 2) — it is **never** subset to the overlap.  
>  
> `engagement_processed.csv` is treated as an **OPTIONAL LEFT JOIN enrichment** — when building the `employee_intelligence` table in Step 16, engagement fields will be `NULL` for the ~50% of employees without a match, and this must be handled explicitly (not silently dropped) in that step.

---
## Part C · Relationship Map

All four table pairs verified below. Findings written to `docs/data_relationships.md`.

---

### C1 · employee_attrition ↔ engagement_processed

In [14]:
# Already computed in Part B — recap
print('Relationship : employee_attrition <-> engagement_processed')
print('Join key     : EmployeeNumber (attrition) = Employee ID (engagement)')
print('Type         : one-to-one where matched (both keys are unique within each file)')
print('Coverage     : 49.7% (731/1470 attrition employees matched)')
print('Status       : CONFIRMED by set intersection in Step 1 (raw) and Step 4 (processed)')

# Confirm both keys are unique in their respective files
att_key_unique = att['EmployeeNumber'].nunique() == len(att)
eng_key_unique = eng['Employee ID'].nunique() == len(eng)
print(f'EmployeeNumber unique in attrition : {att_key_unique}')
print(f'Employee ID unique in engagement   : {eng_key_unique}')

Relationship : employee_attrition <-> engagement_processed
Join key     : EmployeeNumber (attrition) = Employee ID (engagement)
Type         : one-to-one where matched (both keys are unique within each file)
Coverage     : 49.7% (731/1470 attrition employees matched)
Status       : CONFIRMED by set intersection in Step 1 (raw) and Step 4 (processed)
EmployeeNumber unique in attrition : True
Employee ID unique in engagement   : True


### C2 · employee_attrition ↔ occupation_master  (JobRole ↔ Title text match)

In [15]:
att_jobroles   = set(att['JobRole'].dropna().str.strip().unique())
occ_titles     = set(occ['Title'].dropna().str.strip().unique())

matched     = att_jobroles & occ_titles
unmatched   = att_jobroles - occ_titles

print('Relationship : employee_attrition <-> occupation_master')
print('Proposed key : attrition[JobRole] = occupation_master[Title]  (exact text match)')
print()
print(f'Unique JobRole values in attrition : {len(att_jobroles)}')
print(f'Unique Title   values in occ_master: {len(occ_titles)}')
print()
print(f'EXACT MATCHES : {len(matched)} / {len(att_jobroles)}')
print(f'NO MATCH      : {len(unmatched)} / {len(att_jobroles)}')
print()

if matched:
    print('Matched roles:')
    for r in sorted(matched):
        print(f'  + {r}')
    print()

if unmatched:
    print('UNMATCHED roles (will produce NULL O*NET data in role-intelligence step):')
    for r in sorted(unmatched):
        print(f'  ✗ {r}')
    print()
    # Attempt fuzzy partial match to see if near-misses exist
    print('Near-miss check (case-insensitive substring search in occ_master Title):')
    for role in sorted(unmatched):
        role_lower = role.lower()
        candidates = [t for t in occ_titles if role_lower in t.lower() or t.lower() in role_lower]
        if candidates:
            print(f'  [{role}] -- possible matches in occ_master:')
            for c in candidates[:5]:
                print(f'    -> {c}')
        else:
            print(f'  [{role}] -- NO near-miss found in occ_master')
else:
    print('All JobRole values match occupation_master Title exactly.')

Relationship : employee_attrition <-> occupation_master
Proposed key : attrition[JobRole] = occupation_master[Title]  (exact text match)

Unique JobRole values in attrition : 9
Unique Title   values in occ_master: 1016

EXACT MATCHES : 0 / 9
NO MATCH      : 9 / 9

UNMATCHED roles (will produce NULL O*NET data in role-intelligence step):
  ✗ Healthcare Representative
  ✗ Human Resources
  ✗ Laboratory Technician
  ✗ Manager
  ✗ Manufacturing Director
  ✗ Research Director
  ✗ Research Scientist
  ✗ Sales Executive
  ✗ Sales Representative

Near-miss check (case-insensitive substring search in occ_master Title):
  [Healthcare Representative] -- NO near-miss found in occ_master
  [Human Resources] -- possible matches in occ_master:
    -> Human Resources Assistants, Except Payroll and Timekeeping
    -> Human Resources Specialists
    -> Human Resources Managers
  [Laboratory Technician] -- possible matches in occ_master:
    -> Dental Laboratory Technicians
    -> Ophthalmic Laboratory T

### C3 · occupation_master ↔ essential_skills_processed  (O\*NET-SOC Code)

In [16]:
occ_codes  = set(occ['O*NET-SOC Code'].dropna().str.strip())
ess_codes  = set(ess['O*NET-SOC Code'].dropna().str.strip())

occ_in_ess = occ_codes & ess_codes
occ_not_ess = occ_codes - ess_codes
ess_not_occ = ess_codes - occ_codes

print('Relationship : occupation_master <-> essential_skills_processed')
print('Join key     : O*NET-SOC Code (both files)')
print(f'occ_master unique codes   : {len(occ_codes)}')
print(f'ess_processed unique codes: {len(ess_codes)}')
print(f'Codes in occ that match ess  : {len(occ_in_ess)} ({len(occ_in_ess)/len(occ_codes)*100:.1f}% of occ_master)')
print(f'Codes in occ NOT in ess      : {len(occ_not_ess)}')
print(f'Codes in ess NOT in occ      : {len(ess_not_occ)}')

if len(occ_not_ess) > 0:
    print(f'\nOcc codes not in essential_skills (first 10): {sorted(occ_not_ess)[:10]}')
if len(ess_not_occ) > 0:
    print(f'Ess codes not in occ_master (first 10): {sorted(ess_not_occ)[:10]}')

# One-to-many check: each O*NET code in ess maps to multiple skill rows
ess_per_code = ess.groupby('O*NET-SOC Code').size()
print(f'\nSkill rows per O*NET code in essential_skills: '
      f'min={ess_per_code.min()}, max={ess_per_code.max()}, mean={ess_per_code.mean():.1f}')
print('Relationship type: occupation_master (1) <-> essential_skills_processed (many)')
print('Status: CONFIRMED — O*NET-SOC Code is the clean join key')

Relationship : occupation_master <-> essential_skills_processed
Join key     : O*NET-SOC Code (both files)
occ_master unique codes   : 1016
ess_processed unique codes: 910
Codes in occ that match ess  : 910 (89.6% of occ_master)
Codes in occ NOT in ess      : 106
Codes in ess NOT in occ      : 0

Occ codes not in essential_skills (first 10): ['11-1031.00', '11-9039.00', '11-9179.00', '11-9199.00', '13-1199.00', '13-2051.00', '13-2054.00', '13-2099.00', '15-1299.00', '15-2099.00']

Skill rows per O*NET code in essential_skills: min=20, max=20, mean=20.0
Relationship type: occupation_master (1) <-> essential_skills_processed (many)
Status: CONFIRMED — O*NET-SOC Code is the clean join key


### C4 · occupation_master ↔ software_skills_processed  (O\*NET-SOC Code)

In [17]:
sw_codes  = set(sw['O*NET-SOC Code'].dropna().str.strip())

occ_in_sw   = occ_codes & sw_codes
occ_not_sw  = occ_codes - sw_codes
sw_not_occ  = sw_codes - occ_codes

print('Relationship : occupation_master <-> software_skills_processed')
print('Join key     : O*NET-SOC Code (both files)')
print(f'occ_master unique codes   : {len(occ_codes)}')
print(f'sw_processed unique codes : {len(sw_codes)}')
print(f'Codes in occ that match sw   : {len(occ_in_sw)} ({len(occ_in_sw)/len(occ_codes)*100:.1f}% of occ_master)')
print(f'Codes in occ NOT in sw       : {len(occ_not_sw)}')
print(f'Codes in sw NOT in occ       : {len(sw_not_occ)}')

if len(occ_not_sw) > 0:
    print(f'\nOcc codes not in software_skills (first 10): {sorted(occ_not_sw)[:10]}')
if len(sw_not_occ) > 0:
    print(f'Sw codes not in occ_master (first 10): {sorted(sw_not_occ)[:10]}')

sw_per_code = sw.groupby('O*NET-SOC Code').size()
print(f'\nSoftware rows per O*NET code: '
      f'min={sw_per_code.min()}, max={sw_per_code.max()}, mean={sw_per_code.mean():.1f}')
print('Relationship type: occupation_master (1) <-> software_skills_processed (many)')
print('Status: CONFIRMED — O*NET-SOC Code is the clean join key')

Relationship : occupation_master <-> software_skills_processed
Join key     : O*NET-SOC Code (both files)
occ_master unique codes   : 1016
sw_processed unique codes : 923
Codes in occ that match sw   : 923 (90.8% of occ_master)
Codes in occ NOT in sw       : 93
Codes in sw NOT in occ       : 0

Occ codes not in software_skills (first 10): ['11-9039.00', '11-9179.00', '11-9199.00', '13-1199.00', '13-2099.00', '15-1299.00', '15-2099.00', '17-2199.00', '17-3019.00', '17-3029.00']

Software rows per O*NET code: min=1, max=430, mean=34.5
Relationship type: occupation_master (1) <-> software_skills_processed (many)
Status: CONFIRMED — O*NET-SOC Code is the clean join key


---
## Write docs/data_relationships.md

In [18]:
rel_doc = '''# Data Relationships

**Generated by:** `notebooks/04_data_relationships.ipynb`  
**Project:** Enterprise HR AI  
**Seed source:** Cleaning decisions from `notebooks/03_data_cleaning.ipynb`

---

## Table Inventory (processed)

| File | Rows | Key Column | Key Type |
|------|------|-----------|----------|
| `employee_attrition_processed.csv` | 1,470 | `EmployeeNumber` | Unique integer — ANCHOR |
| `engagement_processed.csv` | 2,845 | `Employee ID` | Unique integer |
| `occupation_master.csv` | 1,016 | `O*NET-SOC Code` | Unique code |
| `essential_skills_processed.csv` | 18,200 | `O*NET-SOC Code` | Non-unique (one-to-many) |
| `software_skills_processed.csv` | 31,821 | `O*NET-SOC Code` | Non-unique (one-to-many) |

---

## Relationship 1 — employee_attrition ↔ engagement_processed

| Property | Value |
|----------|-------|
| Left key | `employee_attrition_processed.EmployeeNumber` |
| Right key | `engagement_processed.Employee ID` |
| Join type | LEFT JOIN (attrition is anchor, never subset) |
| Cardinality | One-to-one where matched |
| Coverage | **49.7%** — 731 of 1,470 attrition employees have engagement records |
| Status | **CONFIRMED** (verified on raw files in Step 1, re-verified on processed in Step 4) |

**Decision:**  
`employee_attrition_processed.csv` is the ANCHOR table for all attrition modelling (Day 2).  
`engagement_processed.csv` is OPTIONAL LEFT JOIN enrichment.  
When building `employee_intelligence` (Step 16), engagement fields will be NULL for ~50% of employees — this must be handled explicitly, not silently dropped.

---

## Relationship 2 — employee_attrition ↔ occupation_master

| Property | Value |
|----------|-------|
| Proposed left key | `employee_attrition_processed.JobRole` |
| Proposed right key | `occupation_master.Title` |
| Join type | LEFT JOIN (text match) |
| Cardinality | Many-to-one (many employees per job role) |
| Status | **ASSUMED — exact text match fails for most IBM HR job roles** |

**Gap:** IBM HR dataset job roles (e.g. `Sales Executive`, `Research Scientist`) do NOT match  
O*NET Title strings exactly. A manual mapping table or fuzzy-match lookup is required  
before this join can be used in the role-intelligence step (Day 1 notebook 10).  
See Part C output in notebook 04 for the full match/no-match list.

---

## Relationship 3 — occupation_master ↔ essential_skills_processed

| Property | Value |
|----------|-------|
| Join key | `O*NET-SOC Code` (both files) |
| Join type | One-to-many (one occupation → many skill rows) |
| Status | **CONFIRMED** (key is clean in both files, verified in Step 4) |

---

## Relationship 4 — occupation_master ↔ software_skills_processed

| Property | Value |
|----------|-------|
| Join key | `O*NET-SOC Code` (both files) |
| Join type | One-to-many (one occupation → many software tool rows) |
| Status | **CONFIRMED** (key is clean in both files, verified in Step 4) |

---

## Cleaning Decisions (from Step 3 — seeds for this doc)

- **Age=17 correction (engagement IDs 1743, 2038):** Both corrected to Age=21 using DOB + Survey Date.
  Decision was unambiguous (DOB 2001, Survey 2023). No rows excluded.
- **Whitespace stripping:** Applied to all object columns in all 5 processed files.
  Notable: `DepartmentType` in engagement had 1,910 cells with `Production       ` trailing spaces.
- **Missing values:** Zero missing values in attrition and engagement after Age correction.
  `essential_skills.Not Relevant` has 50% nulls — by design (Importance rows have no Level flag).
- **Duplicates:** Zero exact duplicates dropped from any of the five processed files.
- **Dtype enforcement:** Integer columns cast to nullable `Int64`; date columns parsed to `datetime64`.

---

## Open Issues

1. **JobRole ↔ O*NET Title gap** — IBM HR job roles do not match O*NET Titles exactly.
   Requires a manual/fuzzy mapping table before the role-intelligence step (Day 1 nb 10).
   Action: create `data/external/jobrole_onet_mapping.csv` in Step 5.

2. **Employee ID namespace question** — 49.7% overlap could be coincidental numeric overlap
   rather than shared employees. Business key validation (HR system confirmation) recommended
   before using engagement features in production models.

3. **Employee_Performance_Dataset.csv** — 0% ID overlap with attrition (confirmed Step 1).
   Treated as synthetic/unrelated. Excluded from processed outputs.

4. **employee_performance_pro.csv** — 25.6% overlap, 63.8% missing on CustomerSatisfaction.
   Deferred. May be re-evaluated if engagement coverage remains too low after Step 16.
'''

doc_path = os.path.join(DOCS, 'data_relationships.md')
with open(doc_path, 'w', encoding='utf-8') as f:
    f.write(rel_doc)
print(f'Written: {doc_path}  ({len(rel_doc):,} chars)')

Written: ..\docs\data_relationships.md  (4,558 chars)


---
## Final Processed File Inventory

In [19]:
print('=== data/processed/ after Step 4 ===')
for fname in sorted(os.listdir(PROC)):
    fpath = os.path.join(PROC, fname)
    size  = os.path.getsize(fpath)
    df_check = pd.read_csv(fpath, nrows=0)
    print(f'  {fname:<45s}  {size:>9,} bytes')

=== data/processed/ after Step 4 ===
  employee_attrition_processed.csv                 227,974 bytes
  engagement_processed.csv                         647,282 bytes
  essential_skills_processed.csv                 2,252,736 bytes
  occupation_master.csv                            268,030 bytes


  software_skills_processed.csv                  3,557,545 bytes
